# A05 - Boussinesq 自然对流方腔

这是二维流体传热耦合的核心模型之一。无量纲稳态方程：

- `u_x + v_y = 0`
- `u*u_x + v*u_y + p_x - Pr*(u_xx+u_yy) = 0`
- `u*v_x + v*v_y + p_y - Pr*(v_xx+v_yy) - Ra*Pr*T = 0`
- `u*T_x + v*T_y - (T_xx+T_yy) = 0`

左壁热 `T=1`，右壁冷 `T=0`，上下壁绝热 `T_y=0`，四壁无滑移。

为了 notebook 可运行，默认 `Ra` 较小。提高 `Ra` 会明显增加训练难度。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(105)
device = "cuda" if torch.cuda.is_available() else "cpu"
Pr = 0.71
Ra = 100.0
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=80, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 4)]
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        return self.net(xy)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

model = MLP().to(device)

In [ ]:
def residual(x, y):
    x.requires_grad_(True); y.requires_grad_(True)
    out = model(torch.cat([x, y], dim=1))
    u, v, p, T = out[:,0:1], out[:,1:2], out[:,2:3], out[:,3:4]
    u_x, u_y = grad(u, x), grad(u, y)
    v_x, v_y = grad(v, x), grad(v, y)
    p_x, p_y = grad(p, x), grad(p, y)
    T_x, T_y = grad(T, x), grad(T, y)
    u_xx, u_yy = grad(u_x, x), grad(u_y, y)
    v_xx, v_yy = grad(v_x, x), grad(v_y, y)
    T_xx, T_yy = grad(T_x, x), grad(T_y, y)
    cont = u_x + v_y
    mom_x = u*u_x + v*u_y + p_x - Pr*(u_xx + u_yy)
    mom_y = u*v_x + v*v_y + p_y - Pr*(v_xx + v_yy) - Ra*Pr*T
    energy = u*T_x + v*T_y - (T_xx + T_yy)
    return cont, mom_x, mom_y, energy

def bc_loss(n=160):
    s = torch.rand(n, 1, device=device)
    # no-slip all walls
    walls = [
        torch.cat([torch.zeros_like(s), s], dim=1),
        torch.cat([torch.ones_like(s), s], dim=1),
        torch.cat([s, torch.zeros_like(s)], dim=1),
        torch.cat([s, torch.ones_like(s)], dim=1),
    ]
    loss_uv = sum(torch.mean(model(w)[:,0:2]**2) for w in walls)
    left = model(torch.cat([torch.zeros_like(s), s], dim=1))[:,3:4]
    right = model(torch.cat([torch.ones_like(s), s], dim=1))[:,3:4]
    loss_T_lr = torch.mean((left - 1)**2) + torch.mean(right**2)
    x_top = s.clone().requires_grad_(True)
    y_top = torch.ones_like(x_top).requires_grad_(True)
    T_top = model(torch.cat([x_top, y_top], dim=1))[:,3:4]
    T_y_top = grad(T_top, y_top)
    x_bot = s.clone().requires_grad_(True)
    y_bot = torch.zeros_like(x_bot).requires_grad_(True)
    T_bot = model(torch.cat([x_bot, y_bot], dim=1))[:,3:4]
    T_y_bot = grad(T_bot, y_bot)
    loss_adiabatic = torch.mean(T_y_top**2) + torch.mean(T_y_bot**2)
    p_ref = model(torch.tensor([[0.5, 0.5]], device=device))[:,2:3]
    return loss_uv + loss_T_lr + loss_adiabatic + torch.mean(p_ref**2)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(14000):
    x = torch.rand(1200, 1, device=device)
    y = torch.rand(1200, 1, device=device)
    res = residual(x, y)
    loss_pde = sum(torch.mean(r**2) for r in res)
    loss_bc = bc_loss()
    loss = loss_pde + 20 * loss_bc
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append([loss.item(), loss_pde.item(), loss_bc.item()])
    if epoch % 2500 == 0:
        print(f"epoch={epoch:5d}, total={loss.item():.3e}, pde={loss_pde.item():.3e}, bc={loss_bc.item():.3e}")

In [ ]:
nx = ny = 80
xs = torch.linspace(0, 1, nx, device=device)
ys = torch.linspace(0, 1, ny, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1,1), Y.reshape(-1,1)], dim=1)
with torch.no_grad():
    out = model(xy)
    U = out[:,0].reshape(nx,ny); V = out[:,1].reshape(nx,ny)
    T = out[:,3].reshape(nx,ny)
    speed = torch.sqrt(U**2 + V**2)

plt.figure(figsize=(12,4))
for i, (field, title) in enumerate([(T, "temperature"), (speed, "speed"), (U, "u")], 1):
    plt.subplot(1,3,i)
    plt.imshow(field.cpu().T, origin="lower", extent=[0,1,0,1])
    plt.colorbar(); plt.title(title)
plt.tight_layout(); plt.show()